# Banc d'essai ECOS : quel modèle pour le patient simulé ?

Ce notebook compare des modèles de langage **sur vos stations**, avec le code réel de l'application
(fiche compilée, agent patient et ses outils, arbitre) :

1. pour chaque modèle candidat, il démarre un serveur vLLM sur le GPU de Colab ;
2. il rejoue des consultations figées, identiques pour tous : étudiant **bon**, **maladroit**, **tricheur** ;
3. chaque réplique passe par les contrôles automatiques (mot interdit, jargon, sortie de rôle, langue…) ;
4. un modèle **juge** note chaque réplique (naturel, fait inventé, hors rôle) ;
5. il produit un tableau comparatif et un fichier de **notation à l'aveugle** pour les enseignants.

**Avant de lancer** : menu *Exécution → Modifier le type d'exécution → GPU A100* (L4 pour les petits modèles).
Comptez ~5 unités de calcul par heure d'A100 40 Go ; une comparaison de 2 à 3 modèles prend 1 h 30 à 2 h.

**Données** : utilisez des stations fictives (`exemple`) ou votre pack privé. Le pack et les répliques
passent sur la machine Colab et dans vos résultats : gardez-les dans votre Drive, jamais dans le dépôt.
Aucune donnée d'étudiant réel.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Configuration

- `STATIONS` : `"exemple"` (cas Dupont, fictif), `"televerser"` (vous choisissez votre pack `.zip`),
  ou un chemin sur votre Drive (`/content/drive/MyDrive/...zip`).
- `CANDIDATS` : les modèles à comparer, avec les options vLLM de leur fiche Hugging Face.
  Le nom servi doit contenir « qwen3 » pour que l'application coupe le raisonnement des modèles Qwen3.
- `JUGE` : le modèle qui note toutes les répliques (le même pour tous les candidats).

In [ ]:
BRANCHE = "claude/kind-pasteur-n2utdt"
STATIONS = "exemple"                 # "exemple", "televerser" ou chemin d'un pack sur le Drive
MAX_STATIONS = None                  # ex. 5 pour un premier essai rapide
PROFILS = "bon,maladroit,tricheur"
ORAL = False                         # True : règles du mode vocal
SAUVER_SUR_DRIVE = True
VLLM = "vllm>=0.19.0"                # Qwen3.6 demande vLLM 0.19 ou plus

TOOLS_HERMES = ["--enable-auto-tool-choice", "--tool-call-parser", "hermes"]
TOOLS_QWEN36 = ["--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder", "--reasoning-parser", "qwen3"]

CANDIDATS = [
    # Référence : le modèle actuel de l'application
    {"label": "qwen3-8b", "model": "Qwen/Qwen3-8B",
     "args": ["--max-model-len", "16384", *TOOLS_HERMES]},
    # Candidat : MoE 35B (3B actifs), en 4 bits comme il tiendrait sur la L40S à côté de la voix
    {"label": "qwen3.6-35b-a3b-awq", "model": "cyankiwi/Qwen3.6-35B-A3B-AWQ-4bit", "served": "Qwen/Qwen3.6-35B-A3B",
     "args": ["--max-model-len", "16384", *TOOLS_QWEN36]},
    # Sur A100 80 Go : la version FP8 officielle
    # {"label": "qwen3.6-35b-a3b-fp8", "model": "Qwen/Qwen3.6-35B-A3B-FP8",
    #  "args": ["--max-model-len", "16384", *TOOLS_QWEN36]},
    # Autre candidat : reprendre l'identifiant et les options de sa fiche Hugging Face
    # {"label": "mistral-small", "model": "mistralai/...", "args": ["--tokenizer-mode", "mistral",
    #  "--config-format", "mistral", "--load-format", "mistral", "--enable-auto-tool-choice", "--tool-call-parser", "mistral"]},
]
JUGE = CANDIDATS[1]                  # le plus gros modèle disponible ; None pour ne pas juger
JUGE_REFLEXION = False               # True : le juge raisonne avant de noter (plus lent, plus fiable)

## 2. Installation (5 à 10 min)

In [ ]:
import os, subprocess
if not os.path.isdir("/content/ECOS-CHATBOT"):
    !git clone -q --depth 1 -b {BRANCHE} https://github.com/BSac33/ECOS-CHATBOT.git /content/ECOS-CHATBOT
else:
    !cd /content/ECOS-CHATBOT && git pull -q
!pip install -q "{VLLM}" sqlmodel click
APP = "/content/ECOS-CHATBOT/backend_ecos_chatbot/app"
!cd {APP} && python scripts/bench_patient.py --help | head -3

## 3. Stations et dossier des résultats

In [ ]:
import datetime, os
if STATIONS == "televerser":
    from google.colab import files
    envoi = files.upload()                       # choisissez votre fichier .ecospack.zip
    STATIONS = "/content/" + next(iter(envoi))
if SAUVER_SUR_DRIVE or STATIONS.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
date = datetime.datetime.now().strftime("%Y-%m-%d_%Hh%M")
RES = f"/content/drive/MyDrive/ecos_banc/{date}" if SAUVER_SUR_DRIVE else f"/content/resultats_banc/{date}"
os.makedirs(RES, exist_ok=True)
OPTIONS = ["--stations", STATIONS, "--resultats", RES]
if MAX_STATIONS:
    OPTIONS += ["--max-stations", str(MAX_STATIONS)]
print("Stations :", STATIONS, "| résultats :", RES)

## 4. Outils : démarrer et arrêter un serveur vLLM

In [ ]:
import os, secrets, signal, subprocess, sys, time, urllib.request
CLE = secrets.token_hex(16)      # clé du serveur local, pour reproduire les conditions réelles
os.environ["BENCH_API_KEY"] = CLE

def nom_servi(c):
    return c.get("served") or c["model"]

def demarrer(c, port=8000):
    cmd = ["vllm", "serve", c["model"], "--port", str(port), "--api-key", CLE,
           "--gpu-memory-utilization", "0.90", "--served-model-name", nom_servi(c), *c["args"]]
    journal = f"{RES}/vllm_{c['label']}.log"   # gardé avec les résultats
    proc = subprocess.Popen(cmd, stdout=open(journal, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    debut = time.time()
    while time.time() - debut < 2400:
        if proc.poll() is not None:
            print(open(journal).read()[-3000:])
            raise RuntimeError(f"vLLM s'est arrêté ({c['label']}) : voir {journal}")
        try:
            urllib.request.urlopen(f"http://localhost:{port}/health", timeout=2)
            print(f"{c['label']} prêt en {int(time.time() - debut)} s")
            return proc
        except Exception:
            time.sleep(10)
    raise TimeoutError(f"{c['label']} pas prêt après 40 min : voir {journal}")

def arreter(proc):
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        proc.wait(timeout=120)
    except Exception:
        os.killpg(proc.pid, signal.SIGKILL)
    time.sleep(5)                # le temps de libérer la mémoire du GPU

def banc(*args):
    r = subprocess.run([sys.executable, "scripts/bench_patient.py", *args], cwd=APP)
    if r.returncode:
        raise RuntimeError(f"bench_patient.py {args[0]} a échoué ({r.returncode})")

## 5. Jouer les consultations avec chaque candidat
Le serveur de chaque modèle est démarré puis arrêté. Le premier démarrage d'un modèle comprend
son téléchargement (5 à 20 Go). En cas d'interruption, relancez cette cellule : chaque candidat
recommence ses propres consultations.

In [ ]:
for c in CANDIDATS:
    print(f"\n=== {c['label']} ===")
    proc = demarrer(c)
    try:
        args = ["jouer", "--base-url", "http://localhost:8000/v1", "--model", nom_servi(c),
                "--label", c["label"], "--profils", PROFILS, *OPTIONS]
        banc(*args, *(["--oral"] if ORAL else []))
    finally:
        arreter(proc)

## 6. Faire noter les répliques par le juge
Le juge reçoit la fiche du personnage, la fin de la conversation et la réplique. Il dit si elle est
naturelle, si elle invente un fait médical absent de la fiche, et si elle sort du rôle. Les notes
déjà données sont conservées : la cellule peut être relancée.

In [ ]:
if JUGE:
    proc = demarrer(JUGE)
    try:
        banc("juger", "--base-url", "http://localhost:8000/v1", "--model", nom_servi(JUGE),
             "--concurrence", "16", *(["--reflexion"] if JUGE_REFLEXION else []), *OPTIONS)
    finally:
        arreter(proc)

## 7. Résultats
- **écarts vus %** : répliques avec une sortie de route parmi celles que l'étudiant aurait vues
  (messages laissés passer par l'arbitre). L'objectif est **0**.
- **écarts tous %** : même chose si l'arbitre laissait tout passer (robustesse du personnage seul).
- **naturel %**, **invente %** : notes du juge. À confirmer par les enseignants avec `notation_aveugle.csv`.
- **outil examen %** : l'examen clinique annoncé déclenche bien l'outil de l'examinateur.
- **attaques bloquées %**, **faux blocages %** : qualité de l'arbitre (même modèle que le patient).
- **1er mot (s)** : délai avant le premier mot de la réplique (sur A100 : indicatif pour la L40S).

In [ ]:
banc("resume", "--resultats", RES)
import pandas as pd
display(pd.read_csv(f"{RES}/resume.csv").set_index("modele").T)

## 8. Notation par les enseignants
`notation_aveugle.csv` (dans le dossier des résultats) contient un échantillon de répliques de tous
les modèles, mélangées, sans le nom du modèle. Faites remplir les colonnes `naturel`, `invente`,
`hors_role` (oui/non). La correspondance est dans `notation_cle.csv` : ne la montrez pas aux notateurs.
Pour affirmer « plus de 95 % naturel », visez 300 à 500 répliques notées par modèle
(`resume --echantillon`).

In [ ]:
import shutil
shutil.make_archive("/content/resultats_banc", "zip", RES)
if not SAUVER_SUR_DRIVE:
    from google.colab import files
    files.download("/content/resultats_banc.zip")
print("Résultats :", RES)